# Азбука Морзе – тренажёр: демонстрация

Ноутбук показывает, как работает библиотека `MorseTrainer.Core` и консольное приложение `morse`.

Перед запуском соберите решение из корня репозитория:

```
dotnet build
```

Ноутбук открывается в VS Code с расширением **Polyglot Notebooks** (ядро C#).

## 1. Подключение библиотеки

In [ ]:
#r "../src/MorseTrainer.Core/bin/Debug/net10.0/MorseTrainer.Core.dll"

using MorseTrainer.Core;

## 2. Перевод текста в код и обратно

Буквы разделяются пробелом, слова – ` / `. Регистр не важен.

In [ ]:
var translator = new MorseTranslator();

var encoded = translator.Encode("SOS help");
Console.WriteLine(encoded.Result);
Console.WriteLine(translator.Decode(encoded.Result).Result);

Кириллица. Для одинаковых кодов (А/A, Б/B …) при расшифровке используется выбранный алфавит. `Ё` кодируется как `Е`.

In [ ]:
var cyr = new MorseTranslator(AlphabetKind.Cyrillic);

var privet = cyr.Encode("Привет, ёж 42");
Console.WriteLine(privet.Result);
Console.WriteLine(cyr.Decode(privet.Result).Result);

// Тот же код латиницей
Console.WriteLine(translator.Decode(privet.Result).Result);

Похожие символы (`·`, `•`, `_`, `–`, `—`) приводятся к точке и тире. Слова можно разделять тремя пробелами.

In [ ]:
Console.WriteLine(translator.Decode("··· ——— ···   •— —•").Result);
Console.WriteLine(MorseTranslator.LooksLikeCode("... --- ..."));
Console.WriteLine(MorseTranslator.LooksLikeCode("SOS"));

Неизвестные символы пропускаются, неизвестные комбинации заменяются на `?`. В обоих случаях возвращаются предупреждения.

In [ ]:
var bad = translator.Encode("Hi #1 ~");
Console.WriteLine(bad.Result);
bad.Warnings.ToList().ForEach(Console.WriteLine);

var badCode = translator.Decode(".... ........ ..");
Console.WriteLine(badCode.Result);
badCode.Warnings.ToList().ForEach(Console.WriteLine);

## 3. Тайминги и сигнал

Длина точки – `1200 / WPM` мс. Тире – 3 точки, пауза внутри буквы – 1, между буквами – 3, между словами – 7.

In [ ]:
new[] { 5, 10, 15, 20, 30, 40 }
    .Select(wpm => new MorseTiming(wpm))
    .Select(t => new { WPM = t.Wpm, Точка = t.DotMs, Тире = t.DashMs, МеждуБуквами = t.LetterGapMs, МеждуСловами = t.WordGapMs })

`MorseSignal` – последовательность тонов и пауз. `CharIndex` показывает, к какому символу текста относится элемент.
Слово PARIS вместе с паузой после него занимает ровно 50 единиц.

In [ ]:
var timing = new MorseTiming(20);
var paris = MorseSignal.FromText("PARIS", timing);

Console.WriteLine($"Длительность: {paris.TotalMs} мс + пауза слова {timing.WordGapMs} мс = {(paris.TotalMs + timing.WordGapMs) / timing.DotMs} единиц");
paris.Elements.Take(12)

## 4. Картинка SVG

In [ ]:
var svg = new MorseImageRenderer().Render(translator.Encode("SOS help").Result);
HTML(svg)

Можно выделить букву (так GUI подсвечивает текущую букву при воспроизведении):

In [ ]:
HTML(new MorseImageRenderer().Render(cyr.Encode("Морзе").Result, AlphabetKind.Cyrillic, highlightLetter: 2))

## 5. Звук WAV

WAV (16 бит, моно, 44100 Гц) генерируется вручную. Плеер ниже встроен в ноутбук через `<audio>` с base64.

In [ ]:
var signal = MorseSignal.FromText("CQ CQ", new MorseTiming(18));
byte[] wav = new WavGenerator(frequencyHz: 650, volume: 0.6).Generate(signal);

Console.WriteLine($"Размер файла: {wav.Length} байт, длительность {signal.TotalMs} мс");
HTML($"<audio controls src=\"data:audio/wav;base64,{Convert.ToBase64String(wav)}\"></audio>")

## 6. Передача ключом: `KeyingDecoder`

Заранее заданные нажатия (время в мс) при 20 WPM (точка = 60 мс). Короткое нажатие – точка, длинное – тире,
пауза от 3 единиц заканчивает букву, от 7 – слово.

In [ ]:
var decoder = new KeyingDecoder(20);
decoder.SymbolDecoded += (_, part) => Console.Write($"[{part}]");

// (нажали, отпустили): S = ..., пауза слова, E = .
(int Down, int Up)[] presses = [(0, 50), (120, 170), (240, 290), (800, 850)];
foreach (var (down, up) in presses)
{
    decoder.KeyDown(TimeSpan.FromMilliseconds(down));
    decoder.KeyUp(TimeSpan.FromMilliseconds(up));
}
decoder.Tick(TimeSpan.FromMilliseconds(1100));

Console.WriteLine();
Console.WriteLine($"Код: {decoder.Code}");
Console.WriteLine($"Текст: {translator.Decode(decoder.Code).Result}");

## 7. Консольное приложение

Вызываем собранный `morse.exe`.

In [ ]:
using System.Diagnostics;
using System.IO;

var morseExe = Path.GetFullPath("../src/MorseTrainer.Cli/bin/Debug/net10.0-windows/morse.exe");

// Запускает morse с аргументами и печатает вывод и код возврата
void Morse(params string[] args)
{
    var info = new ProcessStartInfo(morseExe)
    {
        RedirectStandardOutput = true,
        RedirectStandardError = true,
        StandardOutputEncoding = System.Text.Encoding.UTF8,
        StandardErrorEncoding = System.Text.Encoding.UTF8
    };
    foreach (var arg in args)
        info.ArgumentList.Add(arg);

    using var process = Process.Start(info)!;
    string output = process.StandardOutput.ReadToEnd();
    string error = process.StandardError.ReadToEnd();
    process.WaitForExit();

    Console.Write(output);
    Console.Write(error);
    Console.WriteLine($"(код возврата {process.ExitCode})");
}

In [ ]:
Morse("encode", "SOS help");
Morse("decode", "... --- ... / .... . .-.. .--.");
Morse("encode", "привет", "--alphabet", "cyr");
Morse("encode", "A#B", "--strict");

In [ ]:
var outDir = Path.Combine(Path.GetTempPath(), "morse-showcase");
Directory.CreateDirectory(outDir);
var wavPath = Path.Combine(outDir, "cq.wav");
var svgPath = Path.Combine(outDir, "cq.svg");

Morse("encode", "CQ CQ", "--wav", wavPath, "--wpm", "20", "--freq", "700");
Morse("encode", "CQ CQ", "--svg", svgPath);

HTML(File.ReadAllText(svgPath))